# 03 — Li, Liu, Liu & Wei (2024), "Replicating and digesting anomalies in the Chinese A-share market"

All 469 anomalies, four portfolio procedures, seven factor models (lighthouse.md 5.3). Logic lives in `src/replication/li/`; this notebook only calls it and shows ours vs the paper. Run `python -m replication.li --build` first to fill the anomaly caches (aggregates only).

In [ ]:
import pandas as pd
pd.set_option('display.max_rows', 200)
from replication.li import tables as T, catalog as C
from replication.li.__main__ import TABLE_FUNCS
from replication.common.compare import compare
segs = T.li_segments(); orig = segs['original']

In [ ]:
ours = pd.concat([f(orig) for f in TABLE_FUNCS.values()], ignore_index=True)
cmp = compare('li', ours)
scored = cmp[cmp['pass'].notna()]
summary = scored.groupby('table')['pass'].agg(['size', 'sum'])
summary.columns = ['scored', 'within tolerance']
summary

In [ ]:
def show(table, rows=None):
    x = cmp[cmp['table'] == table][['row', 'col', 'paper', 'ours', 'kind', 'pass']].copy()
    if rows: x = x[x['row'].str.match(rows)]
    x['pass'] = x['pass'].map({True: 'ok', False: 'MISS'}).fillna('')
    return x

## Table 1: sample and factors

In [ ]:
show('T1', r'A:(1999|2005|2018)')

In [ ]:
show('T1', r'[BC]:')

## Table 2: microcaps and investment capacity

In [ ]:
show('T2', r'A:')

In [ ]:
show('T2', r'[BCD]:Mainboard-VW')

## Table 3: significance rates

In [ ]:
show('T3', r'A:')

## Table 4: significant anomalies (Mainboard-VW)

In [ ]:
show('T4', r'A:count')

In [ ]:
t4 = show('T4', r'A:'); t4[t4['col'] == 'm'].head(80)

## Table 5: factor models

In [ ]:
show('T5', r'A:')

## Table 6 (excerpt) and Table 7

In [ ]:
show('T6', r'A:sue_q_1')

In [ ]:
show('T7', r'[AC]:Total')

## Text numbers and section 4.2

In [ ]:
show('TXT')

## Extension (lighthouse 6): same functions, later years

The extended and full segments headline the look-ahead-free version (`portfolios.HOLDING_MONTH_FILTER = False`, user decision 2026-10-11); the authors' procedure is shown next to it.

In [ ]:
rows = []
for name, seg in segs.items():
    t3 = T.table3(seg)
    a = t3[(t3['row'] == 'A:Mainboard-VW') & (t3['col'] == 'All categories:SHT')]['value'].iloc[0]
    rows.append({'segment': str(seg), 'share significant (SHT, MB-VW)': round(a, 2)})
pd.DataFrame(rows)